In [44]:
# Imports
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

## Load Cluster Node Profiles

In [45]:
def load_node_profile(node):
    filepath = f'../../tool/infrastructure-profiler/profiles/{node}.csv'

    with open(filepath) as f:
        data = f.readlines()[1].split(',')

    return {
        'ram': int(data[1].strip()),
        'cores': int(data[2].strip()),
        'io_read': float(data[3].strip()),
        'io_write': float(data[4].strip()),
        'sysbench': float(data[5].strip()),
        'z7b': float(data[6].strip()),
        'bench_time': float(data[7].strip())
    }

In [46]:
nodes = ['aws-metal-c5n', 'aws-metal-z1d', 'gpgnode-16', 'gpgnode-22', 'tu-06', 'hu-c40', 'hu-c42']
node_profiles = {}

for node in nodes:
    profile = load_node_profile(node)
    node_profiles[node] = profile

In [61]:
def combine_df(input_df, trace_df):
    rows = []
    assert len(trace_df) == len(input_df['name'].unique())

    for name, group in input_df.groupby('name'):
        no_files = str(len(group))
        filepaths = group['path']
        paths = ', '.join([filepath.split('/')[-1] for filepath in filepaths])
        total_size = group['size'].sum()  # / 1073741824  # B to GiB
        task_record = trace_df[trace_df['name'] == name].iloc[0]
        row = [task_record['task_id'], task_record['name'], task_record['%cpu'], 
               task_record['start'], task_record['complete'], task_record['complete'] - task_record['start'], 
               no_files, paths, total_size]
        rows.append(row)

    combined = pd.DataFrame(rows, columns=['id', 'name', '%cpu', 'start', 'complete', 'runtime', 'no_files', 'files', 'input_size_b'])
    return combined

In [62]:
def load_profiling_runs(node, workflows, samples):
    node_profile = node_profiles[node]
    runs = {}

    for workflow, samples in zip(workflows, samples):
        runs[workflow] = {}

        for sample in samples:
            sample_dir = f'{node}/{workflow}/{sample}/'
            input_df = pd.read_csv(sample_dir + 'input.csv', sep=';')
            trace_df = pd.read_csv(sample_dir + 'trace.csv')
            runs[workflow][sample] = combine_df(input_df, trace_df)

    return (node_profile, runs)

In [63]:
def load_workflow_execution(cluster, workflow, iteration):
    execution_dir = f'../{cluster}/{workflow}/{iteration}/'
    input_df = pd.read_csv(execution_dir + 'input.csv', sep=';')
    trace_df = pd.read_csv(execution_dir + 'trace.csv')
    return combine_df(input_df, trace_df)

## Predict from TU-06

In [64]:
TU_06 = 'tu-06'
TU_06_workflows = ['rnaseq', 'chipseq']
TU_06_samples = [['sample_b', 'sample_c', 'sample_d'], ['sample_e', 'sample_f', 'sample_g', 'sample_h']]
(TU_06_profile, TU_06_runs) = load_profiling_runs(TU_06, TU_06_workflows, TU_06_samples)

In [65]:
HU_40 = 'hu-c40'
HU_40_workflows = ['rnaseq', 'chipseq']
HU_40_samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d'], ['sample_e', 'sample_f', 'sample_g', 'sample_h']]
(HU_40_profile, HU_40_runs) = load_profiling_runs(HU_40, HU_40_workflows, HU_40_samples)

In [68]:
HU_42 = 'hu-c42'
HU_42_workflows = ['rnaseq', 'chipseq']
HU_42_samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d'], ['sample_e', 'sample_f', 'sample_g', 'sample_h']]
(HU_42_profile, HU_42_runs) = load_profiling_runs(HU_42, HU_42_workflows, HU_42_samples)

In [67]:
# identify real executions on hu-cluster
hu_rnaseq = load_workflow_execution('hu-cluster', 'rnaseq', 1)
hu_chipseq = load_workflow_execution('hu-cluster', 'chipseq', 1)

In [75]:
# hu-c40 -- rnaseq sample_d profiling 
comb_rnaseq_sample_d = HU_40_runs['rnaseq']['sample_d']

x = comb_rnaseq_sample_d['input_size_b'].values
y = comb_rnaseq_sample_d['runtime'].values

X_train, X_test, y_train, y_test = train_test_split(x.reshape(-1, 1), y, test_size=0.1, random_state=42)
# q: what are these reshaping / test_size / random_state ? how to use multiple runs i.e. diff samples to create this training dataset?
linear_regression = LinearRegression()
linear_regression.fit(X_train, y_train)

# use model to predict duration based on input size from a FULL WORKFLOW RUN ON
# - the same cluster node type
# - a different cluster node type

# Example of how to use the model to predict duration for a new input size
new_input_size = 1696922320  # value in Bytes
predicted_duration = linear_regression.predict([[new_input_size]])
print(f'Linear Regression predicted duration for input size {new_input_size} bytes: {predicted_duration[0]} seconds')

for _, record in hu_rnaseq.iterrows():
    if record['input_size_b'] > 1696922320:
        print(record['input_size_b'], record['runtime'])

Linear Regression predicted duration for input size 1696922320 bytes: 65441.23653903401 seconds
1696927331 14000
1767700102 14000
2022648072 17000
1828709600 14000
1728736218 17000
2101335401 21000
2308693518 19000
2153929379 20000
1804255323 18000
1894355988 18000
1835491485 16000
1742851244 14000
1990219450 18000
1710502431 16000
1726366250 16000
1702272445 14000
1880465468 17000
1788232977 15000
2166761415 18000
2367705982 19000
1866723707 18000
2114268875 18000
2018934508 17000
2155544080 20000
1796935168 17000
2122200499 20000
2441856775 19000
1807026930 15000
2095075983 19000
1964551674 16000
2556894025 23000
1696927331 0
1767700102 0
2022648072 0
1828709600 1000
1728736218 0
2101335401 1000
2308693518 0
2153929379 0
1804255323 1000
1894355988 0
1835491485 1000
1742851244 0
1990219450 0
1710502431 0
1726366250 0
1702272445 0
1880465468 0
1788232977 0
2166761415 0
2367705982 0
1866723707 0
2114268875 0
2018934508 0
2155544080 0
1796935168 0
2122200499 0
2441856775 0
1807026930 0
2

In [ ]:
# Make a simple svm, linear regression and random forest regression model to predict y from x
# models = {
#     'SVR': SVR(),
#     'RandomForest': RandomForestRegressor(),
#     'LinearRegression': LinearRegression()
# }

# mse = mean_squared_error(y_test, y_pred) 
# print(f'Linear Regression MSE: {mse}')
